In [1]:
import os
import mne
from mne_bids import BIDSPath, write_raw_bids
import gc
import matplotlib.pyplot as plt
import numpy as np
from mne.preprocessing import (
    ICA,
    find_bad_channels_maxwell, 
    annotate_muscle_zscore
)

%matplotlib qt

In [98]:
# Subject information
sid = "sub-20250924zjy"
ses = "ses-01"
run = "run-04"

ds_root = "./data"
cal_file = os.path.join(ds_root, sid, ses, "meg", f"{sid}_{ses}_acq-calibration_meg.dat")
ct_file = os.path.join(ds_root, sid, ses, "meg", f"{sid}_{ses}_acq-crosstalk_meg.fif")

tsss_root = os.path.join(ds_root, "derivatives/meg_derivatives/tsss")
tsss_bids_path = BIDSPath(subject=sid.split('-')[-1], 
                          session=ses.split('-')[-1], 
                          run=run.split('-')[-1], 
                          task="ConjSemProj", datatype="meg", 
                          suffix='meg', extension='.fif',
                          root=tsss_root)

proc_root = os.path.join(ds_root, "derivatives/meg_derivatives/preprocessing")
proc_bids_path = BIDSPath(subject=sid.split('-')[-1], 
                          session=ses.split('-')[-1], 
                          run=run.split('-')[-1], 
                          task="ConjSemProj", datatype="meg", 
                          suffix='meg', extension='.fif',
                          root=proc_root)

raw_fname = os.path.join(ds_root, sid, ses, "meg", f"{sid}_{ses}_task-ConjSemProj_{run}_meg.fif")

### Apply MAXFilter and tSSS

In [ ]:
# Perform tSSS and MAXFilter
raw = mne.io.read_raw_fif(raw_fname, allow_maxshield=True)

# Find all the events.
## However, the stimulus onset was synchronized with the right edge of its marker.
## So the stimulus onset was 0.3 sec after the onset of its marker.
events = mne.find_events(
    raw, 
    stim_channel="STI101",
    output="offset",
    consecutive="increasing"
)

# Additionally, we convert events into annotations.
## "block-j/sent-k/topic-i/stimID-n"
mapping = {}
for i in range(1, 85):
    topic = (i - 1) // 12 + 1
    block = ((i - 1) % 12) // 4 + 1
    sent = ((i - 1) % 12) % 4 + 1
    mapping[i] = f"block-{block}/sent-{sent}/topic-{topic}/stimID-{i}"
mapping[100] = "WaitKeyResponse"
annot_from_events = mne.annotations_from_events(
    events=events,
    event_desc=mapping,
    sfreq=raw.info["sfreq"],
    orig_time=raw.info["meas_date"],
)
raw.set_annotations(annot_from_events)

## Detect bad channels
raw.info["bads"] = []
raw_check = raw.copy()
auto_noisy_chs, auto_flat_chs, auto_scores = find_bad_channels_maxwell(
    raw_check,
    cross_talk=ct_file,
    calibration=cal_file,
    return_scores=True,
    verbose=True,
)

bads = raw.info["bads"] + auto_noisy_chs + auto_flat_chs
bads = bads + ["MEG0113", "MEG0142", "MEG0111"]
bads = list(set(bads))
raw.info["bads"] = bads

del raw_check
gc.collect()

## Spatiotemporal SSS (not include empty-room data)
raw_tsss = mne.preprocessing.maxwell_filter(
    raw, cross_talk=ct_file, calibration=cal_file,
    st_correlation=0.98, st_duration=10
)

## Apply notch filter
raw_tsss.load_data().notch_filter(np.arange(50, 351, 50))

write_raw_bids(
    raw_tsss, 
    bids_path=tsss_bids_path,
    allow_preload=True,
    format="FIF",
    overwrite=True
)

Opening raw data file ./data\sub-20250924zjy\ses-01\meg\sub-20250924zjy_ses-01_task-ConjSemProj_run-04_meg.fif...
    Read a total of 13 projection items:
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
        generated with dossp-2.1 (1 x 306)  idle
    Range : 103000 ... 811999 =    103.000 ...   811.999 secs
Ready.


C:\Users\xuz\AppData\Local\Temp\10\ipykernel_111892\1496104028.py:2: RuntimeWarning: This file contains raw Internal Active Shielding data. It may be distorted. Elekta recommends it be run through MaxFilter to produce reliable results. Consider closing the file and running MaxFilter on the data.
  raw = mne.io.read_raw_fif(raw_fname, allow_maxshield=True)


169 events found on stim channel STI101
Event IDs: [  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  96 100]
Applying low-pass filter with 40.0 Hz cutoff frequency ...
Reading 0 ... 708999  =      0.000 ...   708.999 secs...
Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 331 samples (0.331 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    2.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    3.6s


Scanning for bad channels in 141 intervals (5.0 s) ...
102 of 102 magnetometer types replaced with T3.
    No bad MEG channels
    Processing 204 gradiometers and 102 magnetometers
    Using fine calibration sub-20250924zjy_ses-01_acq-calibration_meg.dat


[Parallel(n_jobs=1)]: Done 306 out of 306 | elapsed:    3.9s finished


        Adjusting non-orthogonal EX and EY
        Adjusted coil orientations by (μ ± σ): 0.3° ± 0.3° (max: 2.1°)
    Automatic origin fit: head of radius 86.3 mm
    Using origin 2.1, 4.6, 45.7 mm in the head frame
        Interval   1:    0.000 -    4.999
        Interval   2:    5.000 -    9.999
        Interval   3:   10.000 -   14.999
        Interval   4:   15.000 -   19.999
        Interval   5:   20.000 -   24.999
        Interval   6:   25.000 -   29.999
        Interval   7:   30.000 -   34.999
        Interval   8:   35.000 -   39.999
        Interval   9:   40.000 -   44.999
        Interval  10:   45.000 -   49.999
        Interval  11:   50.000 -   54.999
        Interval  12:   55.000 -   59.999
        Interval  13:   60.000 -   64.999
        Interval  14:   65.000 -   69.999
        Interval  15:   70.000 -   74.999
        Interval  16:   75.000 -   79.999
        Interval  17:   80.000 -   84.999
        Interval  18:   85.000 -   89.999
        Interval  19:   90.0

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    2.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    3.5s
[Parallel(n_jobs=1)]: Done 306 out of 306 | elapsed:    3.8s finished


Writing 'data\derivatives\meg_derivatives\tsss\participants.tsv'...
Writing 'data\derivatives\meg_derivatives\tsss\participants.json'...
Writing 'data\derivatives\meg_derivatives\tsss\sub-20250924zjy\ses-01\meg\sub-20250924zjy_ses-01_coordsystem.json'...
Writing 'data\derivatives\meg_derivatives\tsss\sub-20250924zjy\ses-01\meg\sub-20250924zjy_ses-01_coordsystem.json'...
The provided raw data contains annotations, but you did not pass an "event_id" mapping from annotation descriptions to event codes. We will generate arbitrary event codes. To specify custom event codes, please pass "event_id".
Used Annotations descriptions: [np.str_('WaitKeyResponse'), np.str_('block-1/sent-1/topic-1/stimID-1'), np.str_('block-1/sent-1/topic-2/stimID-13'), np.str_('block-1/sent-1/topic-3/stimID-25'), np.str_('block-1/sent-1/topic-4/stimID-37'), np.str_('block-1/sent-1/topic-5/stimID-49'), np.str_('block-1/sent-1/topic-6/stimID-61'), np.str_('block-1/sent-1/topic-7/stimID-73'), np.str_('block-1/sent-2/to

BIDSPath(
root: data/derivatives/meg_derivatives/tsss
datatype: meg
basename: sub-20250924zjy_ses-01_task-ConjSemProj_run-04_meg.fif)

### Annotate Muscle Artefacts

In [ ]:
threshold_muscle = 10
annotations_muscle, scores_muscle = annotate_muscle_zscore(
    raw_tsss, 
    ch_type="mag", 
    threshold=threshold_muscle, 
    min_length_good=0.2,
    filter_freq=[110, 140]
)

annotations_event = raw_tsss.annotations 
raw_tsss.set_annotations(annotations_event + annotations_muscle)

raw_tsss.plot(start=50)
plt.show()

del raw
gc.collect()

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1.1e+02 - 1.4e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 110.00
- Lower transition bandwidth: 27.50 Hz (-6 dB cutoff frequency: 96.25 Hz)
- Upper passband edge: 140.00 Hz
- Upper transition bandwidth: 35.00 Hz (-6 dB cutoff frequency: 157.50 Hz)
- Filter length: 121 samples (0.121 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.5s
[Parallel(n_jobs=1)]: Done 102 out of 102 | elapsed:    2.2s finished


Setting up low-pass filter at 4 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 4.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 5.00 Hz)
- Filter length: 1651 samples (1.651 s)



[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


1284

Channels marked as bad:
none


### Apply ICA for Artefact Attenuation

In [101]:
# Filtering
filt_raw = raw_tsss.copy()
filt_raw.load_data().filter(l_freq=1., h_freq=40.)

# Create ICA Object. Use signal after filtering to fit ICA.
ica = ICA(n_components=40, max_iter="auto", random_state=1016)
ica.fit(filt_raw)

raw_tsss.load_data()
ica.plot_sources(raw_tsss, show_scrollbars=True)
ica.plot_components()
plt.show()

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 3301 samples (3.301 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    2.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    3.6s


Fitting ICA to data using 306 channels (please be patient, this may take a while)


[Parallel(n_jobs=1)]: Done 306 out of 306 | elapsed:    3.9s finished


Omitting 16467 of 709000 (2.32%) samples, retaining 692533 (97.68%) samples.
Selecting by number: 40 components
Fitting ICA took 53.0s.
Creating RawArray with float64 data, n_channels=42, n_times=709000
    Range : 103000 ... 811999 =    103.000 ...   811.999 secs
Ready.


In [102]:
# Check exclued components
print("Excluded IC: ", ica.exclude)

# Plot the properties of excluded components
ica.plot_properties(raw_tsss, picks=ica.exclude)
ica.plot_overlay(raw_tsss, exclude=ica.exclude, picks="mag")
plt.show()

Excluded IC:  [0, 1, 3, 5, 13, 24, 39, 28]
    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
329 matching events found
No baseline correction applied
0 projection items activated
Applying ICA to Raw instance
    Transfo

In [103]:
# Apply ICA and check the reconstruction of signals
regexp = r"(MEG[12][45][123]1|EOG00.)"
artifact_picks = mne.pick_channels_regexp(raw_tsss.ch_names, regexp=regexp)

del filt_raw
gc.collect()

reconst_raw = raw_tsss.copy()
ica.apply(reconst_raw)

reconst_raw.plot(order=artifact_picks, n_channels=len(artifact_picks), show_scrollbars=True)
plt.show()

Applying ICA to Raw instance
    Transforming to ICA space (40 components)
    Zeroing out 8 ICA components
    Projecting back using 306 PCA components


Channels marked as bad:
none


In [104]:
# Save
write_raw_bids(
    reconst_raw, 
    bids_path=proc_bids_path,
    allow_preload=True,
    format="FIF",
    overwrite=True
)

del raw_tsss
gc.collect()

Writing 'data\derivatives\meg_derivatives\preprocessing\participants.tsv'...
Writing 'data\derivatives\meg_derivatives\preprocessing\participants.json'...
Writing 'data\derivatives\meg_derivatives\preprocessing\sub-20250924zjy\ses-01\meg\sub-20250924zjy_ses-01_coordsystem.json'...
Writing 'data\derivatives\meg_derivatives\preprocessing\sub-20250924zjy\ses-01\meg\sub-20250924zjy_ses-01_coordsystem.json'...
The provided raw data contains annotations, but you did not pass an "event_id" mapping from annotation descriptions to event codes. We will generate arbitrary event codes. To specify custom event codes, please pass "event_id".
Used Annotations descriptions: [np.str_('BAD_muscle'), np.str_('WaitKeyResponse'), np.str_('block-1/sent-1/topic-1/stimID-1'), np.str_('block-1/sent-1/topic-2/stimID-13'), np.str_('block-1/sent-1/topic-3/stimID-25'), np.str_('block-1/sent-1/topic-4/stimID-37'), np.str_('block-1/sent-1/topic-5/stimID-49'), np.str_('block-1/sent-1/topic-6/stimID-61'), np.str_('blo

35546

### Extract Event-Related Trials

In [105]:
# Exclude key response events
events_id = {}
for i in range(1, 85):
    topic = (i - 1) // 12 + 1
    block = ((i - 1) % 12) // 4 + 1
    sent = ((i - 1) % 12) % 4 + 1
    events_id[f"block-{block}/sent-{sent}/topic-{topic}/stimID-{i}"] = i

# Reject criterion
reject = dict(grad=5000e-13,  # unit: T / m (gradiometers)
              mag=5e-12)      # unit: T (magnetometers)

epochs = mne.Epochs(reconst_raw, 
                    events, events_id,
                    tmin=-0.3, tmax=7.0,
                    reject=reject, 
                    reject_by_annotation=True,
                    preload=True,
                    proj=False)

# Save the epochs
epochs.save(
    os.path.join(
        proc_bids_path.directory,
        proc_bids_path.basename.replace("_meg.fif", "_meg-epo.fif")
    ),
    overwrite=True,
    split_naming="bids"
)

Not setting metadata
84 matching events found
Setting baseline interval to [-0.3, 0.0] s
Applying baseline correction (mode: mean)
Using data from preloaded Raw for 84 events and 7301 original time points ...
15 bad epochs dropped


[WindowsPath('e:/xuz/SemSynProj/Formal-experiment_ver-3/data/derivatives/meg_derivatives/preprocessing/sub-20250924zjy/ses-01/meg/sub-20250924zjy_ses-01_task-ConjSemProj_run-04_meg-epo.fif')]